# Somo 09 - Mfano wa Ubunifu wa Metakognitivi


## Usanidi

Daftari hili linaonyesha mtindo wa kubuni wa Metacognition kwa kutumia Microsoft Agent Framework.

**Mahitaji kabla ya kuanza:**
- Utekelezaji wa Azure OpenAI umewekwa kupitia vigezo vya mazingira
- CLI ya Azure imeidhinishwa (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Metacognition ni nini?

Metacognition ni **kufikiri kuhusu kufikiri**. Katika muktadha wa mawakala wa AI, inamaanisha kujenga mawakala wanaoweza:

- **Kujitafakari** juu ya matokeo yao wenyewe na mchakato wao wa hoja
- **Kutambua makosa** na kurekebisha kwa utulivu badala ya kushindwa kimya
- **Kutathmini** ikiwa majibu yao ni kamili na ya msaada
- **Kubadilisha** mkakati wao wakati mbinu ya awali haifanyi kazi (kwa mfano, kurudi kwenye mfumo wa ziada)

Wakala anayejitafakari si wa kujibu maswali tu — anafuatilia utendaji wake mwenyewe na kurekebisha papo kwa papo.


## Zana za Msingi na za Akiba

Mfano wa kawaida wa metacognition ni **mkakati wa kutegemea zana za akiba**. Wakala anajaribu zana ya msingi kwanza; ikiwa itashindikana (kwa mfano, kosa la 404), wakala anatambua kushindwa na kwa uwazi anahamia kwenye zana ya akiba.

Hii inaakisi mifumo ya ulimwengu halisi ambapo huduma za msingi zinaweza kuwa hazipatikani na wakala lazima wajibainishe tatizo kabla ya kuchagua njia mbadala.

Hapa chini tunaelezea zana mbili za kutafuta ndege:
- **Msingi** — inafunika Paris, Tokyo, na Barcelona
- **Akiba** — inafunika Berlin, Sydney, na New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Wakala Anayejiangalia Mwenyewe na Urejeshaji wa Makosa

Wakala hapa chini ameagizwa kujaribu kwanza mfumo mkuu wa ndege, kutambua matatizo, na kwa uwazi kurudi kwenye mfumo wa chelezo. Baada ya kila jibu, anajitathmini kwa ufupi ikiwa alijibu swali la mtumiaji kikamilifu.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Muundo wa Kujitathmini

Sehemu nyingine ya ufahamu juu ya utambuzi ni **kujitathmini**: wakala tofauti (au wakala huyo huo katika upitio wa pili) hupitia jibu kwa ukamilifu, usahihi, na kuwa wa msaada.

Hapo chini tunaunda wakala wa `ResponseEvaluator` anayetoa alama kwa majibu ya wakala wa usafiri katika vigezo vitatu.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Muhtasari

Katika somo hili ulijifunza jinsi ya kujenga **mawakala wa metakognitifu** kwa kutumia Microsoft Agent Framework:

- **Utafakari wa nafsi**: Mawakala wanaofuatilia mchakato wao wa kufikiri na kuwaeleza kwa uwazi kilichotokea.
- **Urejeshaji wa makosa kwa kutumia fallbacks**: Mfumo wa zana ya msingi + chelezo ambapo wakala hugundua kushindwa (kwa mfano, makosa 404) na kwa otomatiki anajaribu chanzo mbadala.
- **Tathmini ya nafsi**: Wakala mtathmini tofauti anayetoa alama kwa majibu kwa vigezo vya ukamilifu, usahihi, na jinsi yanavyosaidia.

Mifumo hii inafanya mawakala kuwa imara zaidi, wazi zaidi, na wa kuaminika — sifa muhimu kwa utekelezaji katika mazingira ya uzalishaji.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Onyo:
Nyaraka hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI Co-op Translator (https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kuhakikisha usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au kutokuwa sahihi. Nyaraka ya asili katika lugha yake ya asili inapaswa kuzingatiwa kama chanzo chenye mamlaka. Kwa taarifa muhimu, inapendekezwa kutumia tafsiri ya kitaalamu inayofanywa na mtafsiri wa binadamu. Hatuwajibiki kwa kutokuelewana au tafsiri zisizo sahihi zinazotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
